# Multi-Agent Financial Analysis System

AAI-520 Final Team Project

## 1. Project Overview and GitHub Repository

This notebook is the canonical project entry point. It demonstrates the reusable research workflow implemented under `src/` without redefining application logic in notebook cells.

## 2. Setup and Configuration

This section locates the repository, imports the public workflow interface, and sets the ticker. It does not contact Yahoo Finance or Ollama; those calls begin in the end-to-end section.

In [2]:
import os
import sys
from pathlib import Path

from IPython.display import JSON, Markdown, display


def find_project_root(start: Path) -> Path:
    """Find the repository whether Jupyter starts at its root or notebooks/."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.graph import build_research_workflow
from src.reporting import render_console_summary

workflow = build_research_workflow(project_root=PROJECT_ROOT)
TICKER = os.getenv("DEFAULT_TICKER", "AAPL").strip().upper()

print(f"Project root: {PROJECT_ROOT}")
print(f"Research ticker: {TICKER}")

Project root: /Users/brandonwirgau/Projects/AAI/520/Final Project
Research ticker: AAPL


## 3. Agent Design and Shared State

`ResearchWorkflow` coordinates planner, evaluator, synthesis, and memory components. Each stage adds a typed artifact to `ResearchState`, allowing the notebook and tests to inspect the same results.

## 4. Agent Functions, Data Sources, and Tool Use

The planner chooses from the registered tools. The executor invokes provider-independent tools, currently backed by the Yahoo Finance adapter, and records failures without stopping unrelated tool calls.

## 5. Workflow 1 — Prompt Chaining

The implemented research chain passes structured artifacts through planning, collection, deterministic validation, reflection, synthesis, report validation, and memory curation. The news-specific ingest-to-summary pipeline remains a separate extension point under `src/workflows/`.

## 6. Workflow 2 — Routing

Tool selection is allow-listed against the registry before execution. Specialist content routing can be added behind the existing router interface without changing this notebook entry point.

## 7. Workflow 3 — Evaluator–Optimizer

The evaluator combines deterministic checks with an LLM quality reflection, then validates the synthesized report for coverage and unsafe language. The optimizer module is scaffolded for a future feedback-driven revision loop.

## 8. Memory and Learning Across Runs

After a successful run, the memory curator stores a short timestamped lesson. The planner can retrieve recent notes for the same symbol, but current tool data always remains the source of market evidence.

## 9. End-to-End Investment Research Example

The next cell is the project's live execution point. It requires a running Ollama service with the configured model and network access for Yahoo Finance. Change `TICKER` above, then run this section.

In [3]:
result = workflow.run(TICKER, progress=print)

1/7 Planning the research run
2/7 Collecting market and financial evidence
Running tool: cash_flow
Running tool: company_info
Running tool: financials
Running tool: price_data
3/7 Validating tool observations
4/7 Reflecting on evidence quality
5/7 Synthesizing the research report
6/7 Validating the research report
7/7 Saving lessons for a future run


### Inspect Structured Workflow Artifacts

In [4]:
print(render_console_summary(result))
display(JSON({
    "plan": result["plan"],
    "validation": result["validation"],
    "reflection": result["reflection"],
    "report_validation": result["report_validation"],
    "memory_entry": result.get("memory_entry"),
}))

FINAL RESEARCH REPORT

Apple Inc. (AAPL)
----------------------------------------------------------------------

1. COMPANY OVERVIEW
   Sector:              Technology
   Industry:            Consumer Electronics
   Country:             United States
   Current Price:       $338.98
   Market Cap:          $4.947T
   Enterprise Value:    $4.969T

2. PRICE PERFORMANCE
   One-Year Return:     32.86%
   Annualized Volatility: 24.56%
   Maximum Drawdown:    -13.80%

3. VALUATION
   Trailing P/E:        38.92
   Forward P/E:         35.35
   Price/Sales (TTM):   10.60
   Profit Margin:       27.62%
   Operating Margin:    32.62%
   Return on Equity:    148.75%
   Beta:                1.085

4. FINANCIAL PERFORMANCE (2025)
   Revenue:             $416.161B
   Operating Income:    $133.050B
   Net Income:          $112.010B
   EBITDA:              $144.748B
   Diluted EPS:         $7.46
   EBITDA / Net Income: 1.292x

5. CASH FLOW (2025)
   Operating Cash Flow: $111.482B
   Free Cash Flow:    

<IPython.core.display.JSON object>

### Final Research Report

In [5]:
display(Markdown(result["report"]))

# Apple Inc. Investment Research Report

## Company Overview

### Description
Apple Inc. is a multinational technology company headquartered in Cupertino, California.

### Key Statistics

* Market Capitalization: $494.7 trillion
* Enterprise Value: $496.9 trillion
* Current Price: $338.98
* Industry: Consumer Electronics
* Sector: Technology

## Price Performance

### Historical Data
The company's stock has shown significant growth over the past year, with a 1-year return of 32.86%.

### Volatility
The annualized volatility of the stock is 24.56%, indicating moderate price fluctuations.

## Valuation

### Price-to-Earnings Ratio
The trailing P/E ratio is 38.918484, while the forward P/E ratio is 35.352398. These values are subject to change and may not accurately reflect the company's future performance.

## Financial Performance

### Revenue
The company's total revenue has been steadily increasing over the past few years, with a recent value of $416.16 billion.

### Profitability
Apple's operating income has been consistently high, with a recent value of $133.05 billion.

## Cash Flow

### Operating Cash Flow
The company's operating cash flow has been increasing over the past few years, with a recent value of $111.482 billion.

### Free Cash Flow
The company's free cash flow has been decreasing in recent years, with a value of $98.767 billion in 2023. This is a suspicious value and warrants further investigation.

### Capital Expenditure
Capital expenditure data is limited, and no breakdown by segment is available. This makes it difficult to assess the impact of capital expenditure on cash flow.

## Risks and Uncertainties

### Potential Risks
The company's dependence on a few key products (e.g., iPhone) makes it vulnerable to fluctuations in demand and competition.

### Uncertainties
The company's cash flow strategy and capital expenditure strategy are not well understood, making it difficult to assess potential risks and uncertainties.

## Data Quality

### Strengths
Cash flow data is available for the specified periods.

### Weaknesses
Limited information on capital expenditure and its impact on cash flow.

### Missing Information
Detailed analysis of risks and uncertainties, breakdown of capital expenditure by segment.

### Suspicious Values
Low free cash flow in 2023 ($98.767 billion) compared to previous years.

## Further Research

* Investigation into the impact of low free cash flow on Apple's ability to invest in new technologies.
* Analysis of changes in capital expenditure strategy.
* Correlation between capital expenditure and free cash flow.
* Detailed analysis of risks and uncertainties affecting Apple Inc.

## 10. Evaluation, Limitations, and Conclusions

The workflow surfaces deterministic validation issues and model-generated quality feedback for inspection. Current limitations include reliance on one implemented market-data provider, a local model, and a single-pass report; news routing and iterative optimization remain future extensions. Outputs are research support, not personalized investment advice.